# 09 — Longer horizons: 3, 6, 12 months ahead

One-month-ahead forecasting is exhausted (04, 07, 08): persistence MAE 153 is unbeaten by nine variants. The remaining forecasting question is whether structure pays at **longer horizons**, where persistence has more time to be wrong and seasonality/drift/flows have more room to accumulate.

Two design points:

1. **Splits are by *target* month, not origin month.** An Apr-2024 origin with a 12-month horizon lands in the sealed year — splitting by origin would silently validate on FY 2025-26. Instead: training rows have targets before Apr 2024, validation rows have targets inside FY 2024-25, and an assert proves no target ever reaches the sealed year. Because the window is defined by target month, the 1-month row count differs slightly from 04's 1,407 (12 target months here vs 11 origins there) — the fresh persistence MAE is printed as the anchor.
2. **Uncertainty is calibrated and checked on *different* years.** The 90% band around the persistence forecast is calibrated on FY 2023-24 targets and its coverage measured on FY 2024-25 — calibrating and scoring on the same rows would return 90% by construction and prove nothing.

Cohort, features, seed and the sealed year are all as in 07/08.

In [1]:
import pandas as pd                          # tables
import numpy as np                           # array arithmetic
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor   # gradient-boosted trees, NaN-native

PROC = Path("../data/processed")
TEST_START = pd.Timestamp("2025-04-01")      # sealed test year - no target may reach it
VAL_START = pd.Timestamp("2024-04-01")       # validation targets: FY 2024-25
CAL_START = pd.Timestamp("2023-04-01")       # conformal calibration targets: FY 2023-24
COVID = pd.Timestamp("2020-03-01")           # regime break: first lockdown month
SEED = 42                                    # the one declared seed

panel = pd.read_csv(PROC / "rtt_kh03_canonical_panel.csv", parse_dates=["month"])
flows = pd.read_csv(PROC / "rtt_flows_panel.csv", parse_dates=["month"])
dev = panel[panel["month"] < TEST_START]     # development window, Apr 2018 - Mar 2025

# the 04/07/08 cohort with the full 08 feature set
df = dev[(dev["tfunc_code"] == "C_110") & dev["fl_client"] & dev["is_public"]].copy()
df = df.sort_values(["provider_code", "month"]).reset_index(drop=True)

g = df.groupby("provider_code")["total_waiting"]
for k in (1, 2, 3):
    df[f"delta_{k}"] = df["total_waiting"] - g.shift(k)   # momentum, as in 07/08
df["month_of_year"] = df["month"].dt.month

df = df.merge(flows[flows["tfunc_code"] == "C_110"].drop(columns="tfunc_code"),
              on=["provider_code", "month"], how="left")
df["net_flow"] = df["referrals"] - df["completed"]
df["net_flow_3m"] = df.groupby("provider_code")["net_flow"].transform(lambda s: s.rolling(3).mean())

FEATS = ["total_waiting", "pct_gt18", "pct_gt52", "delta_1", "delta_2", "delta_3",
         "month_of_year", "occupancy_ga", "referrals", "completed", "net_flow", "net_flow_3m"]
g = df.groupby("provider_code")["total_waiting"]         # regrouped on the merged frame's index
print(df["provider_code"].nunique(), "providers,", len(df), "provider-months")

131 providers, 10562 provider-months


## All horizons, all models

Per horizon *h*: the target is the list *h* months later; every model forecasts it from the origin month's features. References: **persistence** (target = origin level), **same-month-last-year** (the actual list 12 months before the target — at *h* = 12 this *is* persistence, so it is skipped), **drift** (origin level + training-mean *h*-month change), and the **gbm on the *h*-month change** with the full 08 feature set. Drift and gbm run in both declared regimes.

In [2]:
# Goal: one results table - MAE/RMSE per (horizon, model, regime) on FY 2024-25 targets.
frames, results = {}, []

def rec(h, model, regime, err):              # err = truth minus forecast, one value per row scored
    results.append({"h": h, "model": model, "regime": regime, "mae": np.abs(err).mean(),
                    "rmse": np.sqrt((err ** 2).mean()), "n": len(err)})

for h in (1, 3, 6, 12):
    d = df.copy()
    d["target"] = g.shift(-h)                # the list h reported months later (0.1% of steps
                                             # span a reporting gap - measured in 07, tolerated)
    d["target_month"] = d["month"] + pd.DateOffset(months=h)
    d["seasonal"] = g.shift(12 - h)          # the actual list 12 months before the target month
    d = d.dropna(subset=["target"])          # origins whose target would fall past Mar 2025
    assert d["target_month"].max() < TEST_START   # a sealed-year target would trip this
    frames[h] = d

    tr = d[d["target_month"] < VAL_START]    # targets up to Mar 2024
    va = d[d["target_month"] >= VAL_START]   # targets inside FY 2024-25
    truth = va["target"].to_numpy()

    rec(h, "persistence", "-", truth - va["total_waiting"].to_numpy())
    if h != 12:                              # at h=12 same-month-last-year IS persistence
        s = va["seasonal"].notna().to_numpy()     # provider entered less than 12-h months before origin
        rec(h, "seasonal", "-", truth[s] - va["seasonal"].to_numpy()[s])
    for reg, t in [("full", tr), ("post", tr[tr["month"] >= COVID])]:
        rec(h, "drift", reg, truth - (va["total_waiting"] + (t["target"] - t["total_waiting"]).mean()).to_numpy())
        m = HistGradientBoostingRegressor(random_state=SEED)
        m.fit(t[FEATS], t["target"] - t["total_waiting"])
        rec(h, "gbm-delta+flows", reg, truth - (va["total_waiting"].to_numpy() + m.predict(va[FEATS])))

res = pd.DataFrame(results)
for h in (1, 3, 6, 12):
    print(f"--- {h}-month ahead ---")
    for _, r in res[res["h"] == h].sort_values("mae").iterrows():
        print(f"  {r['model']:16s} {r['regime']:5s} MAE {r['mae']:7,.0f}   RMSE {r['rmse']:7,.0f}   (n={r['n']})")

--- 1-month ahead ---
  persistence      -     MAE     154   RMSE     326   (n=1534)
  gbm-delta+flows  full  MAE     156   RMSE     327   (n=1534)
  drift            full  MAE     157   RMSE     326   (n=1534)
  gbm-delta+flows  post  MAE     158   RMSE     332   (n=1534)
  drift            post  MAE     160   RMSE     327   (n=1534)
  seasonal         -     MAE     742   RMSE   1,133   (n=1524)
--- 3-month ahead ---
  persistence      -     MAE     317   RMSE     558   (n=1532)
  drift            full  MAE     331   RMSE     560   (n=1532)
  gbm-delta+flows  full  MAE     333   RMSE     564   (n=1532)
  gbm-delta+flows  post  MAE     338   RMSE     572   (n=1532)
  drift            post  MAE     345   RMSE     567   (n=1532)
  seasonal         -     MAE     742   RMSE   1,133   (n=1524)
--- 6-month ahead ---
  persistence      -     MAE     501   RMSE     799   (n=1529)
  drift            full  MAE     538   RMSE     817   (n=1529)
  gbm-delta+flows  full  MAE     552   RMSE     842 

## Uncertainty: a conformal band around persistence

Whatever the horizon, persistence is the forecast the project currently trusts most. For the allocation phase the useful add-on is a *calibrated* error band: the 90th percentile of |actual − persistence| over FY 2023-24 targets becomes the band half-width; its honesty is then measured as empirical coverage on FY 2024-25 — a year the calibration never saw.

In [3]:
# Goal: per horizon, the 90% persistence band and its out-of-year coverage, stored as a table
# so the figure notebook can draw it without re-fitting anything.
bands = []
for h in (1, 3, 6, 12):
    d = frames[h]
    cal = d[(d["target_month"] >= CAL_START) & (d["target_month"] < VAL_START)]   # FY 2023-24 targets
    va = d[d["target_month"] >= VAL_START]                                        # FY 2024-25 targets
    q90 = np.quantile(np.abs(cal["target"] - cal["total_waiting"]), 0.90)
    cover = (np.abs(va["target"] - va["total_waiting"]) <= q90).mean()
    bands.append({"h": h, "half_width": q90, "coverage": cover, "n_cal": len(cal), "n_val": len(va)})
    print(f"{h:2d}-month: 90% band = ±{q90:,.0f} patients (calibrated on {len(cal)} FY 2023-24 rows)"
          f"   coverage on FY 2024-25: {cover:.1%} of {len(va)} rows")

bands = pd.DataFrame(bands)
# a band calibrated on one year and scored on the next has no guarantee of landing on 90%;
# this assert fires if a rerun ever drifts far enough to stop being a usable planning input
assert bands["coverage"].between(0.85, 0.92).all()
bands.to_csv(PROC / "val_conformal_bands_c110.csv", index=False)
print(bands.shape, "->", PROC / "val_conformal_bands_c110.csv")

 1-month: 90% band = ±298 patients (calibrated on 1524 FY 2023-24 rows)   coverage on FY 2024-25: 89.0% of 1534 rows
 3-month: 90% band = ±649 patients (calibrated on 1524 FY 2023-24 rows)   coverage on FY 2024-25: 88.7% of 1532 rows
 6-month: 90% band = ±1,053 patients (calibrated on 1519 FY 2023-24 rows)   coverage on FY 2024-25: 87.6% of 1529 rows
12-month: 90% band = ±1,714 patients (calibrated on 1512 FY 2023-24 rows)   coverage on FY 2024-25: 89.6% of 1524 rows
(4, 5) -> ..\data\processed\val_conformal_bands_c110.csv


In [4]:
# Goal: store the full results table so these numbers stay re-readable without a rerun.
res.to_csv(PROC / "val_horizons_c110.csv", index=False)
print(res.shape, "->", PROC / "val_horizons_c110.csv")

(23, 6) -> ..\data\processed\val_horizons_c110.csv
